In [ ]:
# Path configuration
baseline <- '..'  # Firm tables saved by the root notebooks.
inputs <- 'data/alliance'
out <- 'outputs/alliance'


# Alliance regressions and corporate-group sensitivity

Run from `revision_notebooks/` using an R kernel with `arrow`, `data.table` and **fixest 0.12.1**. Set paths in the first cell. Read `complement_reg.parquet` from `../7.2_complement_firm_ctry.ipynb` and `data/alliance/observed_judgments.csv` from notebook 11.

Fit the SI S6 models on firm–candidate pairs, excluding pairs of a firm with itself. Cluster standard errors by focal firm and candidate organization. Repeat the fixed-effects model after excluding pairs classified as belonging to the same corporate group, then also exclude uncertain cases.


In [ ]:
library(arrow)
library(data.table)
library(fixest)
dir.create(out, recursive=TRUE, showWarnings=FALSE)

## Read the firm–candidate regression sample


In [ ]:
d <- as.data.table(read_parquet(file.path(baseline, 'complement_reg.parquet'),
  col_select=c('supply','demand','copatgreen','n_gbb','samecntry',
               'log_supply_totalpat','log_demand_totalpat')))
d <- d[supply != demand]

## Baseline specifications

The fixed-effects model contains log GBB complementarity and same country, with focal and candidate effects. Set `fixef.rm="none"` for the baseline and ownership fits. In fixest 0.12.1, `fixef.K="nested"` counts nonnested fixed effects.


In [ ]:
ssc_rule <- ssc(adj=TRUE, fixef.K='nested', fixef.force_exact=FALSE,
                cluster.adj=TRUE, cluster.df='min', t.df='min')
formulas <- list(
  gbb=copatgreen ~ log1p(n_gbb),
  country=copatgreen ~ log1p(n_gbb) + samecntry,
  size=copatgreen ~ log1p(n_gbb) + samecntry + log_supply_totalpat + log_demand_totalpat,
  fixed_effects=copatgreen ~ log1p(n_gbb) + samecntry | supply + demand)
# Keep the same fixed-effect sample rule for baseline and ownership fits.
# This has no effect on the first three specifications, which have no fixed effects.
models <- list(); coefficients <- list(); metrics <- list()
fit <- function(id, formula, data) {
  m <- feols(formula, data, cluster=~supply+demand, ssc=ssc_rule, fixef.rm="none", lean=TRUE, mem.clean=TRUE)
  models[[id]] <<- m
  ct <- coeftable(m); ci <- confint(m)
  coefficients[[id]] <<- data.frame(model=id, term=rownames(ct), estimate=ct[,1],
    std_error=ct[,2], statistic=ct[,3], p_value=ct[,4], conf_low=ci[,1], conf_high=ci[,2], row.names=NULL)
  metrics[[id]] <<- data.frame(model=id, nobs=nobs(m), events=sum(data$copatgreen),
    r2=as.numeric(fitstat(m,'r2')[[1]]), within_r2=as.numeric(fitstat(m,'wr2')[[1]]))
}
for (id in names(formulas)) fit(id, formulas[[id]], d)

## Exclude pairs with possible corporate-group ties

The classifications cover co-assignments between companies in different countries. Apply each classification to both directions of the pair. Re-estimate the model on the full regression sample after excluding (1) same-group pairs and (2) same-group plus uncertain pairs.


In [ ]:
# Generated by notebook 11 from the observed ownership cohort and model responses.
audit <- fread(file.path(inputs, 'observed_judgments.csv'))
d[, ownership_class := 'not_flagged']
directed <- rbind(audit[, .(supply=psn_id_low, demand=psn_id_high, classification)],
                  audit[, .(supply=psn_id_high, demand=psn_id_low, classification)])
d[directed, on=.(supply,demand), ownership_class := i.classification]
fit('drop_same_group', formulas$fixed_effects, d[ownership_class != 'same_group'])
fit('drop_same_and_uncertain', formulas$fixed_effects,
    d[!ownership_class %in% c('same_group','uncertain')])

## Save estimates and manuscript tables

In [ ]:
fwrite(rbindlist(coefficients), file.path(out,'coefficients.csv'))
fwrite(rbindlist(metrics), file.path(out,'model_metrics.csv'))
writeLines(as.character(etable(models[names(formulas)], tex=TRUE)), file.path(out,'baseline.tex'))
writeLines(as.character(etable(models[c('fixed_effects','drop_same_group','drop_same_and_uncertain')],
                               tex=TRUE)), file.path(out,'ownership_exclusions.tex'))
writeLines(capture.output(sessionInfo()), file.path(out,'session_info.txt'))

etable(models)